# Section 2: VPC & Networking

## VPC Configuration

| Setting | Value |
|---------|-------|
| VPC Name | 25xrvl-vpc |
| CIDR Block | 10.0.0.0/16 |
| Region | us-east-1 |
| DNS Hostnames | Enabled |

---

## Subnet Design

| Subnet | CIDR | Type | Availability Zone |
|--------|------|------|-------------------|
| 25xrvl-vpcc-subnet-public1-us-east-1a | 10.0.1.0/24 | Public | us-east-1a |
| 25xrvl-vpcc-subnet-private1-us-east-1a | 10.0.2.0/24 | Private | us-east-1a |
| 25xrvl-emr-1a | 10.0.3.0/24 | Private | us-east-1a |

---

## Internet Gateway

- **Name**: 25xrvl-igw
- **Purpose**: Enable outbound internet access

---

## Route Table

| Destination | Target |
|-------------|--------|
| 10.0.0.0/16 | Local |
| 0.0.0.0/0 | igw-xxxxx |

---

## Security Groups

### EC2 Security Group (25xrvl-ec2-sg)

| Port | Protocol | Source | Purpose |
|------|----------|--------|---------|
| 22 | TCP | 0.0.0.0/0 | SSH |
| 80 | TCP | 0.0.0.0/0 | HTTP |
| 443 | TCP | 0.0.0.0/0 | HTTPS |
| 11434 | TCP | 0.0.0.0/0 | Ollama API |
| 3000 | TCP | 0.0.0.0/0 | OpenWebUI |

### EMR Security Group (25xrvl-emr-sg)

| Port | Protocol | Source | Purpose |
|------|----------|--------|---------|
| 22 | TCP | 25xrvl-ec2-sg | SSH from EC2 |
| 8443 | TCP | 25xrvl-ec2-sg | EMR master |


---

## Terraform Configuration

In [3]:
%%writefile ../../src/infrastructure/main.tf
# VPC Configuration
resource "aws_vpc" "25xrvl_vpc" {
  cidr_block           = "10.0.0.0/16"
  enable_dns_hostnames = true
  enable_dns_support  = true

  tags = {
    Name = "25xrvl-vpc"
  }
}

# Internet Gateway
resource "aws_internet_gateway" "25xrvl_igw" {
  vpc_id = aws_vpc.25xrvl_vpc.id

  tags = {
    Name = "25xrvl-igw"
  }
}

# Public Subnet
resource "aws_subnet" "25xrvl_public" {
  vpc_id                  = aws_vpc.25xrvl_vpc.id
  cidr_block             = "10.0.1.0/24"
  availability_zone      = "us-east-1a"
  map_public_ip_on_launch = true

  tags = {
    Name = "25xrvl-public-1a"
  }
}

# Private Subnet for EMR
resource "aws_subnet" "25xrvl_private" {
  vpc_id                  = aws_vpc.25xrvl_vpc.id
  cidr_block             = "10.0.2.0/24"
  availability_zone      = "us-east-1a"
  map_public_ip_on_launch = false

  tags = {
    Name = "25xrvl-private-1a"
  }
}

# EMR Subnet
resource "aws_subnet" "25xrvl_emr" {
  vpc_id                  = aws_vpc.25xrvl_vpc.id
  cidr_block             = "10.0.3.0/24"
  availability_zone      = "us-east-1a"
  map_public_ip_on_launch = false

  tags = {
    Name = "25xrvl-emr-1a"
  }
}

# Public Route Table
resource "aws_route_table" "25xrvl_public_rt" {
  vpc_id = aws_vpc.25xrvl_vpc.id

  route {
    cidr_block = "0.0.0.0/0"
    gateway_id = aws_internet_gateway.25xrvl_igw.id
  }

  tags = {
    Name = "25xrvl-public-rt"
  }
}

# Associate Public Subnet with Route Table
resource "aws_route_table_association" "public_assoc" {
  subnet_id       = aws_subnet.25xrvl_public.id
  route_table_id = aws_route_table.25xrvl_public_rt.id
}

# EC2 Security Group
resource "aws_security_group" "25xrvl_ec2_sg" {
  name        = "25xrvl-ec2-sg"
  description = "Security group for EC2 instance"
  vpc_id      = aws_vpc.25xrvl_vpc.id

  ingress {
    from_port   = 22
    to_port     = 22
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 443
    to_port     = 443
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 11434
    to_port     = 11434
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 3000
    to_port     = 3000
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  egress {
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }

  tags = {
    Name = "25xrvl-ec2-sg"
  }
}

# EMR Security Group
resource "aws_security_group" "25xrvl_emr_sg" {
  name        = "25xrvl-emr-sg"
  description = "Security group for EMR cluster"
  vpc_id      = aws_vpc.25xrvl_vpc.id

  ingress {
    from_port       = 22
    to_port         = 22
    protocol        = "tcp"
    security_groups = [aws_security_group.25xrvl_ec2_sg.id]
  }

  ingress {
    from_port       = 8443
    to_port         = 8443
    protocol        = "tcp"
    security_groups = [aws_security_group.25xrvl_ec2_sg.id]
  }

  egress {
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }

  tags = {
    Name = "25xrvl-emr-sg"
  }
}

Overwriting ../../src/infrastructure/main.tf


---

## Screenshots Required

> Take the following screenshots for submission:

### 1. Terraform Code
- Open `src/infrastructure/main.tf` in VS Code
- Screenshot showing all Terraform resources
- 02-terraform-01
![02-terraform-01](02-terraform-01.png)
- 02-terraform-02
![02-terraform-02](02-terraform-02.png)

### 2. AWS VPC Console
- Go to: **AWS Console → VPC → Your VPCs**
- Screenshot showing 25xrvl-vpc (CIDR: 10.0.0.0/16)
- 02-vpc-console
![02-vpc-console](02-vpc-console.png)

### 3. Subnets
- Go to: **VPC → Subnets**
- Screenshot showing all 3 subnets:
  - 25xrvl-vpcc-subnet-public1-us-east-1a
 (10.0.1.0/24)
  - 25xrvl-vpcc-subnet-private1-us-east-1a (10.0.2.0/24)
  - 25xrvl-emr-1a (10.0.3.0/24)
  - 02-subnets
![02-subnets](02-subnets.png)
